# Recetario del pipeline FutBot MX — usa lo que ya está empaquetado

**Para qué sirve este notebook:** mostrar **todas las funciones útiles** de `src/` y
cómo llamarlas desde tus propios notebooks, para **no rehacer cosas desde cero**. Casi
todo el pipeline (segmentar, trackear, lotes, overlays, métricas) ya está hecho y se usa
con una o dos líneas.

**Cómo leerlo:**
- Las celdas **ligeras** (rutas, config, registros, conteo de frames, selección de
  videos) corren **sin GPU** — puedes ejecutarlas tal cual.
- Las celdas **pesadas** (cargan SAM3 / corren inferencia) están dentro de
  `if RUN_HEAVY:` y **no se ejecutan** por defecto: el código queda visible para
  copiar. Ponlas en marcha en el **pod (GPU)** cambiando `RUN_HEAVY = True`.

> Regla de oro: **una llamada por video** vía `run_inference`. Baja a las piezas de
> bajo nivel solo cuando necesites control fino.

## 0 — Setup

In [ ]:
# `import src` funciona desde cualquier cwd gracias a `pip install -e .`.
import json
from pathlib import Path

from src.utils import PROJECT_ROOT, get_abs_path, show_frames

RUN_HEAVY = False  # ponlo en True SOLO en el pod (GPU) para correr SAM3/inferencia.
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_HEAVY:", RUN_HEAVY, "(las celdas que cargan SAM3 se saltan si es False)")


def find_a_video():
    # Devuelve un .MOV de ejemplo (ruta project-relative) o None si no hay datos.
    env = {}
    envp = PROJECT_ROOT / ".env"
    if envp.exists():
        for ln in envp.read_text().splitlines():
            ln = ln.strip()
            if ln and not ln.startswith("#") and "=" in ln:
                k, _, v = ln.partition("=")
                env[k.strip()] = v.strip()
    cfg_name = env.get("CONFIG_FILENAME")
    if not cfg_name:
        return None
    cfg = json.loads(get_abs_path(f"configs/{cfg_name}").read_text())
    ddir = PROJECT_ROOT / cfg.get("working_dirs", {}).get("dataset_dir", "")
    movs = sorted({*ddir.rglob("*.MOV"), *ddir.rglob("*.mov")}) if ddir.exists() else []
    return movs[0].relative_to(PROJECT_ROOT) if movs else None


SAMPLE_VIDEO = find_a_video()
print("video de ejemplo:", SAMPLE_VIDEO)

## 1 — Config y rutas (`src.utils`)

Todo es **config-driven**: nunca hardcodees rutas absolutas. `get_abs_path` resuelve una
ruta relativa contra la raíz del proyecto. Las **clases** (robots, pelota, …) viven en el
config y el pipeline las usa solo; no hace falta pasarlas.

In [ ]:
# Cargar el config activo y ver sus clases (las usa el pipeline automáticamente).
env_cfg = (PROJECT_ROOT / ".env").read_text() if (PROJECT_ROOT / ".env").exists() else ""
cfg_name = next(
    (l.split("=", 1)[1].strip() for l in env_cfg.splitlines()
     if l.strip().startswith("CONFIG_FILENAME")),
    None,
)
config = json.loads(get_abs_path(f"configs/{cfg_name}").read_text()) if cfg_name else {}
print("config:", cfg_name)
print("clases:", [c["name"] for c in config.get("classes", [])])
print("outputs_dir:", config.get("working_dirs", {}).get("outputs_dir"))
print("frame_quota:", config.get("preprocess", {}).get("frame_quota"))

## 2 — Las piezas intercambiables (detectores y trackers)

El pipeline compone **detector → tracker**. Los nombres válidos salen de registros; para
añadir uno nuevo, registras una función (no cambias el pipeline).

In [ ]:
from src.core.detectors import _DETECTORS, get_detector  # registro + resolutor
from src.core.trackers import KNOWN_TRACKERS, get_tracker  # noqa: F401

# Detectores y trackers disponibles (introspeccionados de los registros).
print("detectores:", list(_DETECTORS))
print("trackers:", list(KNOWN_TRACKERS))
# get_detector('nombre') / get_tracker('nombre') resuelven la pieza; nombre invalido -> ValueError.

## 3 — Punto de entrada recomendado: `run_inference`

**Una sola función por video.** `mode="segmentation"` (per-frame, `obj_id` inestable) o
`mode="tracking"` (`obj_id` estables). Devuelve siempre
`{"json", "video", "index"}`.

Parámetros que más vas a usar:
- `detector` / `tracker` — eligen las piezas (`None` = default del config).
- `include_masks=True` — embebe máscaras COCO-RLE en el JSON.
- `run_label="mi_experimento"` — manda las salidas a
  `outputs/inference/<run_label>/<stem>/…` (no se pisan entre experimentos).
- `progress=True` — barra de progreso por video.
- `render_video=False` — solo JSON (más rápido; útil en lotes).

In [ ]:
from src.core.inference import run_inference

if RUN_HEAVY and SAMPLE_VIDEO is not None:
    # Segmentación per-frame con YOLO->SAM3, con máscaras, en su propia carpeta.
    seg = run_inference(
        SAMPLE_VIDEO,
        mode="segmentation",
        detector="yolo_sam3",
        include_masks=True,
        run_label="demo/seg",
        progress=True,
    )
    print("segmentación ->", seg["json"])

    # Tracking con SAM3-texto + ByteTrack (obj_id estables).
    trk = run_inference(
        SAMPLE_VIDEO,
        mode="tracking",
        detector="sam3_text",
        tracker="bytetrack",
        max_frames=300,          # acota frames contiguos (videos largos)
        include_masks=True,
        run_label="demo/trk",
        progress=True,
    )
    print("tracking ->", trk["json"], "| índice obj_id->Track en trk['index']")
else:
    print("[RUN_HEAVY=False] código visible arriba; ponlo en True en el pod para correrlo.")

## 4 — Lotes: `run_batch`

Para **varios videos**: carga SAM3 **una sola vez**, salta lo ya hecho (`skip-done`),
aísla errores (un video que falla no detiene el lote) y devuelve un **resumen por
video** con timing (`fps`, `peak_vram_mb`). Selecciona por `split` (0=reserva,
1=fine-tuning, 2=testing) o por lista explícita de rutas/ids.

In [ ]:
from src.core.batch import run_batch

if RUN_HEAVY:
    summary = run_batch(
        mode="tracking",
        split=2,                 # o videos=[...]
        detector="yolo_sam3",
        tracker="botsort",
        max_frames=500,
        include_masks=True,
        run_label="demo/batch",  # salidas y skip-done por config
        progress=True,
    )
    for r in summary:
        print(r["ruta"], "->", r["status"], "| fps:", r.get("fps"))
else:
    print("[RUN_HEAVY=False] run_batch(mode=..., split=2, detector=..., tracker=..., run_label=...)")

## 5 — Bloques de bajo nivel (control fino)

Cuando quieras armar algo propio frame a frame. **Carga SAM3 una sola vez** y reusa el
`bundle`. La currency compartida es `Detection(obj_id, mask, score)`.

In [ ]:
# --- Frames: estos NO necesitan GPU y corren ya ---
from src.core.frame_extraction import (
    extract_frames,      # (N,H,W,3) en memoria: cuota o all_frames
    get_frame_count,     # nº total de frames (metadata, barato)
    get_frame_indices,   # índices fuente muestreados
    get_video_fps,       # fps real del video
    iter_frames,         # generador en streaming (un frame a la vez) -> tracking
)

if SAMPLE_VIDEO is not None:
    print("fps:", round(get_video_fps(SAMPLE_VIDEO), 2))
    print("frames totales:", get_frame_count(SAMPLE_VIDEO))
    frames = extract_frames(SAMPLE_VIDEO)            # cuota (memoria acotada)
    print("cuota:", frames.shape)
    # show_frames(frames)  # <- vista en grilla (descomenta en un notebook con display)
else:
    print("sin video de ejemplo; estas funciones aceptan ruta relativa o absoluta.")

In [ ]:
# --- Detección por frame: SÍ necesita GPU (SAM3) ---
from src.core.sam3_loader import load_sam3
from src.core.segmentation import detect_classes_in_frame  # {clase: [Detection]}

if RUN_HEAVY and SAMPLE_VIDEO is not None:
    bundle = load_sam3()                      # carga ÚNICA; reúsala en todos los frames
    frame = extract_frames(SAMPLE_VIDEO)[0]
    dets = detect_classes_in_frame(frame, classes=None, bundle=bundle)  # classes=None -> config
    for name, ds in dets.items():
        print(f"{name}: {len(ds)} detecciones")
    # Detección por estrategia explícita (igual que usa el pipeline):
    # det_fn = get_detector('yolo_sam3'); det_fn(frame, classes=None, bundle=bundle)
else:
    print("[RUN_HEAVY=False] load_sam3() + detect_classes_in_frame(frame, bundle=bundle)")

## 6 — Overlays y escritura de video

- `overlay_detections(frame, dets_by_class, classes)` — compone un frame anotado
  (color **por clase**). `show_overlay` es solo para mostrar.
- `write_video(array, path, fps)` — escribe un mp4 de un `(N,H,W,3)`.
- `render_obj_id_overlay(json_tracking)` — **post-pase** que, a partir de un JSON de
  tracking, dibuja caja + `nombre #id` + trayectoria por `obj_id` (sin re-inferir).

In [ ]:
from src.core.overlay import overlay_detections, show_overlay  # noqa: F401
from src.core.track_overlay import render_obj_id_overlay
from src.core.video_writer import open_video_writer, write_video  # noqa: F401

if RUN_HEAVY and SAMPLE_VIDEO is not None:
    # Hacer visible el tracking: a partir del JSON de tracking del paso 3.
    out_mp4 = render_obj_id_overlay(trk["json"], trajectory_window=30)
    print("overlay por obj_id ->", out_mp4)
else:
    print("[RUN_HEAVY=False] render_obj_id_overlay('<json_tracking>', trajectory_window=30)")

## 7 — Dónde caen las salidas y cómo se nombran

`inference_paths(stem, outputs_dir, namespace)` arma la ruta canónica por video. Con
`namespace` (= `run_label`) cada config va a su subcarpeta.

In [ ]:
from src.core.inference_schema import inference_paths

outdir = config.get("working_dirs", {}).get("outputs_dir", "outputs")
print("sin run_label:", inference_paths("MI_VIDEO", outdir)[0])
print("con run_label:", inference_paths("MI_VIDEO", outdir, namespace="demo/trk")[0])
# El JSON sigue el esquema común (header auto-descriptivo + frames [+ rle] [+ tracks]).

## 8 — Benchmark sin ground-truth (`src.eval`)

Para comparar configuraciones sobre videos de testing elegidos con semilla.
`benchmark_videos` corre sin GPU; el resto agrega los JSON que produce `run_batch`.

In [ ]:
from src.eval.benchmark import (
    aggregate_config,    # resumen de run_batch -> una fila por config
    benchmark_videos,    # selección reproducible de N videos de testing
    comparison_table,    # filas -> DataFrame con columnas ordenadas
    video_metrics,       # métricas sin-GT de un JSON ya cargado
    write_table,         # DataFrame -> CSV
)

print("5 videos del benchmark (seed=36):")
for v in benchmark_videos(n=5, seed=36):
    print("  ", v)
# Patrón típico: row = aggregate_config(label, run_batch(..., run_label=label));
#                comparison_table([row1, row2, ...])

## 9 — Cheatsheet

| Quiero… | Llamo a… |
|---|---|
| Segmentar/trackear **un video** | `run_inference(video, mode=..., detector=..., tracker=..., run_label=...)` |
| Procesar **muchos videos** | `run_batch(mode=..., split=2 \| videos=[...], run_label=...)` |
| **Ver** el tracking | `render_obj_id_overlay(json_tracking)` |
| Cargar SAM3 **una vez** | `bundle = load_sam3()` y pásalo como `bundle=` |
| Detectar **un frame** | `detect_classes_in_frame(frame, bundle=bundle)` o `get_detector(name)` |
| Leer frames | `extract_frames` (lote) / `iter_frames` (streaming) / `get_frame_count` / `get_video_fps` |
| Resolver una ruta del config | `get_abs_path("data/raw/...")` |
| Comparar configs | `benchmark_videos` + `run_batch(run_label=...)` + `aggregate_config` |

**Aísla siempre tus experimentos con `run_label`** para que las salidas no se pisen, y
usa `progress=True` para ver el avance. La inferencia necesita **GPU** (pod).